## Background

This notebook try to convert old CoT data that was collected from another project and turn them into training samples for the coherent classifier model.

All the data were collected with a multi-step MDP style using GPT-4 and DeepSeek-R1 models (code not in this repo). We need to concatenate the multi-step completions, add add a binary label (default positive).

In [1]:
# simple hack to support import module from parent directory
import sys
sys.path.append('../src')

In [2]:
import random
random.seed(42)

import json
from transformers import AutoTokenizer
from rl4llm.utils import load_from_jsonl_file, save_to_jsonl_file

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:
source_configs = [
    {
        "input_file": "../data/cot_data/gsm_gpt4_episodes_3k.jsonl.gz",
        "source": "gsm",
        "model_name": "gpt4",
    },
    {
        "input_file": "../data/cot_data/math_gpt4_episodes.jsonl.gz",
        "source": "math",
        "model_name": "gpt4",
    },
]

In [5]:
test_file = source_configs[0]['input_file']
loaded_samples = load_from_jsonl_file(test_file)
print(len(loaded_samples))

3167


In [6]:
def pretty_print(item):
    print(json.dumps(item, indent=2))

In [7]:
pretty_print(loaded_samples[0])

{
  "question": "The ratio of boys to girls in a family is 5:7. The total number of children in the family is 180. If the boys are given $3900 to share, how much money does each boy receive?",
  "ground_truth": "52",
  "task_type": "GSM",
  "short_answer": "\\$52",
  "graded_reward": 1.0,
  "transitions": [
    {
      "state_id": "reasoning",
      "state": [
        {
          "role": "system",
          "content": "You're a thoughtful problem solver who thinks deeply about challenges. When exploring problems, let your mind wander naturally by thinking freely and share your authentic inner dialogue as thoughts emerge. Don't worry about following specific steps or formats - just let your genuine reasoning and insights flow, including any uncertainties, connections, or realizations you experience along the way.\nThink of it as having a conversation with yourself as you work through things. Share your internal process \u2013 reasoning, reflections, and creative ideas \u2013 naturally, 

In [8]:
print(len(loaded_samples[0]['transitions']))

3


In [9]:

pretty_print(loaded_samples[0]['transitions'])

[
  {
    "state_id": "reasoning",
    "state": [
      {
        "role": "system",
        "content": "You're a thoughtful problem solver who thinks deeply about challenges. When exploring problems, let your mind wander naturally by thinking freely and share your authentic inner dialogue as thoughts emerge. Don't worry about following specific steps or formats - just let your genuine reasoning and insights flow, including any uncertainties, connections, or realizations you experience along the way.\nThink of it as having a conversation with yourself as you work through things. Share your internal process \u2013 reasoning, reflections, and creative ideas \u2013 naturally, without predefined steps or format. Use \"I\" statements to describe your thinking.\n\n**Import Latex Rules:**\nLatex (inline): \\( \\)\nLatex (block): \\[ \\]\n\nHere are examples of good **Thoughts and Reasoning**:\n\n---\n\n**Example 1:**\n\n**Question:**\nGrandma gave Bryce and Carter some raisins. Bryce received 

In [10]:
def convert_sample(item: dict, model_name: str) -> dict:
    """
    Converts a sample item into a dictionary containing prompt, completion, and token counts.

    Args:
        item (dict): A dictionary containing the question and transitions.

    Returns:
        dict: A dictionary with prompt, completion, and token counts.
    """
    try:
        # Extract the prompt
        prompt = item['question']

        num_turns = len(item['transitions'])

        # Handle transitions to construct the completion
        if num_turns == 3:
            reason_p1 = item['transitions'][0]['action']['text']
            reason_p2 = item['transitions'][1]['action']['text']
            answer = item['transitions'][2]['action']['text']
            completion = f"<think>{reason_p1}\n\n{reason_p2}</think><answer>{answer}</answer>"
        elif num_turns == 2:
            reason = item['transitions'][0]['action']['text']
            answer = item['transitions'][1]['action']['text']
            completion = f"<think>{reason}</think><answer>{answer}</answer>"
        elif num_turns == 1:
            completion = item['transitions'][0]['action']['text']
        else:
            raise ValueError(f"Unexpected number of transitions {num_turns} in the item.")

        # Tokenize the prompt and completion
        prompt_tokens = tokenizer.encode(prompt, return_tensors="pt").squeeze()
        completion_tokens = tokenizer.encode(completion, return_tensors="pt").squeeze()

        # Return the result as a dictionary
        return {
            "question": item['question'],
            "task_type": item['task_type'],
            "completion": completion,
            "model_name": model_name,
            "prompt_tokens": len(prompt_tokens),
            "completion_tokens": len(completion_tokens),
            "total_tokens": len(prompt_tokens) + len(completion_tokens)
        }
    except KeyError as e:
        raise KeyError(f"Missing expected key in item: {e}")
    except Exception as e:
        raise RuntimeError(f"Error processing item: {e}")

In [11]:
converted_samples = []
for item in source_configs:
    loaded_samples = load_from_jsonl_file(item['input_file'])
    print(f"Loaded {len(loaded_samples)} samples from {item['input_file']}")
    samples = [convert_sample(d, item['model_name']) for d in loaded_samples]
    print(f"Finished converting samples for {item['input_file']}")
    converted_samples.extend(samples)
    

Loaded 3167 samples from ../data/cot_data/gsm_gpt4_episodes_3k.jsonl.gz
Finished converting samples for ../data/cot_data/gsm_gpt4_episodes_3k.jsonl.gz
Loaded 4841 samples from ../data/cot_data/math_gpt4_episodes.jsonl.gz
Finished converting samples for ../data/cot_data/math_gpt4_episodes.jsonl.gz


In [12]:
pretty_print(random.choice(converted_samples))

{
  "question": "The measures of the sides of a triangle are 7, 24 and 25. What is the area of the triangle?",
  "task_type": "MATH",
  "completion": "Alright, let's take a look at this problem. The first thing that jumps out at me is the numbers 7, 24, and 25. They seem familiar. Ah, yes, they form a Pythagorean triple. That means the triangle with these side lengths is a right triangle. \n\nThe Pythagorean theorem states that in a right triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides. In this case, $25^2 = 7^2 + 24^2$, which confirms that this is indeed a right triangle.\n\nNow, the area of a triangle is given by the formula $1/2 \\times \\text{base} \\times \\text{height}$. In a right triangle, we can take any of the two sides that form the right angle as the base and the height. So, in this case, we could take 7 and 24 as the base and the height.\n\nSo, the area of the 

In [13]:
output_file = "../data/cot_data/mixed_gsm_math_positive_samples_gpt4.jsonl.gz"
save_to_jsonl_file(converted_samples, output_file)